In [1]:
import importlib
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# -------------------- Dark Theme --------------------
display(HTML("""
<style>
    body, .jp-Notebook, .jp-OutputArea-output, .jp-RenderedHTMLCommon {
        background-color: #1e1e1e !important;
        color: #d4d4d4 !important;
    }
    h2, h3, h4 {
        color: #4fc3f7 !important;
        border-bottom: 2px solid #3498db !important;
        padding-bottom: 4px;
    }
    b, strong { color: #f48fb1 !important; }
    .highlight {
        background-color: #2d2d2d !important;
        padding: 10px;
        border-left: 4px solid #3498db;
        margin: 4px 0;
        color: #d4d4d4;
    }
    code {
        background-color: #333 !important;
        color: #ffcc80 !important;
        padding: 2px 4px;
        border-radius: 4px;
    }
    .dataframe {
        background-color: #2d2d2d !important;
        color: #d4d4d4 !important;
    }
</style>
"""))

# -------------------- Verbosity Flags --------------------
SHOW_VERBOSE = True
SHOW_INFO = True
SHOW_CRITICAL = True
SHOW_DEBUG = True

# -------------------- Helper functions for display --------------------
def display_title(title: str):
    display(HTML(f"<h2>{title}</h2>"))

def display_info(message: str):
    display(HTML(f"<div class='highlight'>{message}</div>"))

print("Environment ready. Dark theme applied.")

Environment ready. Dark theme applied.


In [2]:
# =============================================================================
# CELL 2 — CANONICAL DATASET DISCOVERY (drop-in, no registry module)
# =============================================================================

from pathlib import Path
import importlib
import Extraction as EX
importlib.reload(EX)

display_title("Processed dataset discovery")

DATASETS, CSV_HASHES = EX.discover_processed_datasets(
    datasets_root=EX.PROCESSED_DATASETS_ROOT,
    show_info=True,
)

display_info(
    f"<b>{len(DATASETS)}</b> processed datasets discovered under "
    f"<code>{EX.PROCESSED_DATASETS_ROOT}</code>"
)

# Visual summary
summary = pd.DataFrame([
    {
        "name": name,
        "rows": len(df),
        "columns": ", ".join(df.columns),
        "csv_sha256": CSV_HASHES[name][:16] + "…",
        "sample_label": str(df["label"].iloc[0]),
    }
    for name, df in DATASETS.items()
])
display(summary)

# Contract checks
for name, df in DATASETS.items():
    assert list(df.columns) == list(EX.PROCESSED_COLUMNS), f"{name}: bad columns"
    assert df.index.is_unique and df.index[0] == 0 and df.index[-1] == len(df) - 1, \
        f"{name}: index must be a clean RangeIndex so row i ↔ hidden_states[i]"
    assert df["label"].map(lambda x: isinstance(x, list) and len(x) > 0).all(), \
        f"{name}: all labels must be non-empty Python lists"

display_info("✅ All datasets satisfy the extraction contract.")

for name, df in DATASETS.items():
    display_title(f"Dataset: {name}")
    display_info(f"Shape: {df.shape}")
    display(df.head(2))
    # We'll rely on the probe's column detection later
    display_info(f"Samples: <b>{len(df):,}</b>")

display_title("Unified ID Scheme")
display_info("""
Each sample is assigned a unique integer ID (0..N-1) matching its row index.
This ID links:
• Input text (original DataFrame index)
• Hidden state vector (row in hidden_states.npy)
• Label (row in labels.npy)
No shuffling occurs, guaranteeing one‑to‑one mapping.
""")

In [ ]:
import Probe as probe # How to use the classes of the datasets we don't know about yet ? 
# How should we separate them
# is it wise to change the target labels ? into a polar decision ?? 
## no this would beat the purpose of my project, emotion recognition is not sentiment scoring or polarity check
## it's best to gather different datasets with different classes and various representation of emotions
## the old 7 emotion style and the go emotion's 28 class emotion datasets could be significant help but they are not stand alone
## I need several datasets to use them to compare models and datasets together via the probe results 
# We have to somehow save a hint about the dataset's schema and available classes and other metadata about datasets
## which is extreamly hard when dealing with a new datasets from a url that you don't even know the number of classes or rows ?? 
# maybe it is possible to gather all values under their target together in a set and observe the absolute values which remain 
# would be all the available classes for our datasets, maybe it is possible to use this very same approach to distinguish 
# text and target columns from one another, detecting the text column isn't that hard but the targets could be challenging 


GOEMOTIONS_CLASSES = probe.GOEMOTIONS_CLASSES
ISEAR_CLASSES = probe.ISEAR_CLASSES

goemotions_contract = probe.DatasetContract(
    target_type="goemotions",
    text_column="auto",
    label_column="auto",
    id_column="auto",
    task_type="multi_label",
    class_order=GOEMOTIONS_CLASSES,
    require_provenance=True,
    require_label_fingerprint=True,
    lenient_provenance=False,
    allow_missing_label_fingerprint=True,
)

isear_contract = probe.DatasetContract(
    target_type="isear",
    text_column="auto",
    label_column="auto",
    id_column="auto",
    task_type="single_label",
    class_order=ISEAR_CLASSES,
    require_provenance=True,
    require_label_fingerprint=True,
    lenient_provenance=False,
    allow_missing_label_fingerprint=True,
)

custom_single_label_contract = probe.DatasetContract(
    target_type="custom",
    text_column="auto",
    label_column="auto",
    id_column="auto",
    task_type="single_label",
    class_order=None,
    require_provenance=True,
    require_label_fingerprint=True,
    lenient_provenance=False,
    allow_missing_label_fingerprint=True,
)

DATASET_CONTRACTS = {
    "goemo": goemotions_contract,
    "isear": isear_contract,
    "emotion": custom_single_label_contract,
    "tweet_eval_emotion": custom_single_label_contract,
    "sst2": custom_single_label_contract,
    "amazon_polarity": custom_single_label_contract,
}

probes = [
    probe.ProbeSpec(
        name="linear_logistic",
        type="logistic",
        complexity="linear",
        standardize=True,
        C=1.0,
        max_iter=3000,
        selection_metric="macro_f1",
    ),

    probe.ProbeSpec(
        name="mlp_1_hidden",
        type="mlp",
        complexity="1_hidden",
        standardize=True,
        hidden_dims=["0.5d"],
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=80,
        batch_size=256,
        patience=12,
        selection_metric="macro_f1",
    ),

    probe.ProbeSpec(
        name="mlp_2_hidden",
        type="mlp",
        complexity="2_hidden",
        standardize=True,
        hidden_dims=["0.5d", "0.25d"],
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=80,
        batch_size=256,
        patience=12,
        selection_metric="macro_f1",
    ),

    probe.ProbeSpec(
        name="mlp_3_hidden",
        type="mlp",
        complexity="3_hidden",
        standardize=True,
        hidden_dims=["0.5d", "0.25d", "0.125d"],
        learning_rate=1e-3,
        weight_decay=1e-4,
        epochs=80,
        batch_size=256,
        patience=12,
        selection_metric="macro_f1",
    ),
]

# Probes definition (same as before)
probes = [
    probe.ProbeSpec(name='linear_logistic', type='logistic', complexity='linear',
                    standardize=True, C=1.0, max_iter=3000, selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_1_hidden', type='mlp', complexity='1_hidden',
                    standardize=True, hidden_dims=['0.5d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_2_hidden', type='mlp', complexity='2_hidden',
                    standardize=True, hidden_dims=['0.5d', '0.25d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
    probe.ProbeSpec(name='mlp_3_hidden', type='mlp', complexity='3_hidden',
                    standardize=True, hidden_dims=['0.5d', '0.25d', '0.125d'], learning_rate=1e-3,
                    weight_decay=1e-4, epochs=80, batch_size=256, patience=12,
                    selection_metric='macro_f1'),
]
print('Contracts and probes defined (auto columns, lenient provenance).')

In [ ]:
from pathlib import Path
import json
import pandas as pd

# Match Probe.py's roots exactly.
AMIRALI_MOUNT      = Path("/Volumes/Amirali")
HIDDEN_STATES_ROOT = AMIRALI_MOUNT / "hidden_states"
PROBE_ROOT         = AMIRALI_MOUNT / "probe"


def discover_extraction_pairs(hidden_states_root: Path = HIDDEN_STATES_ROOT) -> pd.DataFrame:
    """
    Walk <hidden_states>/<slug>/<dataset>/ and return every (model, dataset)
    pair that has a completed extraction. This is the input to run_matrix.
    """
    rows = []
    if not hidden_states_root.is_dir():
        return pd.DataFrame(columns=["model", "dataset", "artifact_dir"])

    for model_dir in sorted(hidden_states_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            meta_path = dataset_dir / "extraction.json"
            if not (dataset_dir / "hidden_states.npy").is_file() or not meta_path.is_file():
                continue
            try:
                meta = json.loads(meta_path.read_text())
            except Exception:
                continue
            rows.append({
                # Prefer the HF id recorded by Extraction.py over the folder slug.
                "model":        meta.get("model", {}).get("name") or model_dir.name,
                "dataset":      meta.get("dataset", {}).get("name") or dataset_dir.name,
                "artifact_dir": str(dataset_dir),
            })
    return pd.DataFrame(rows)


def discover_probe_runs(probe_root: Path = PROBE_ROOT) -> pd.DataFrame:
    """
    Walk <probe>/<slug>/<dataset>/index.json and return every run that
    Probe.py has registered there.
    """
    rows = []
    if not probe_root.is_dir():
        return pd.DataFrame(columns=["model", "dataset", "run_key", "trial_hash",
                                     "probes", "results_csv", "task_type", "n_classes"])

    for model_dir in sorted(probe_root.iterdir()):
        if not model_dir.is_dir() or model_dir.name.startswith("_"):
            continue
        for dataset_dir in sorted(model_dir.iterdir()):
            if not dataset_dir.is_dir():
                continue
            index = dataset_dir / "index.json"
            if not index.is_file():
                continue
            try:
                payload = json.loads(index.read_text())
            except Exception:
                continue
            for entry in payload.get("runs", []):
                results_csv = dataset_dir / entry["results_csv"]
                rows.append({
                    "model":       entry["model"],
                    "dataset":     entry["dataset"],
                    "run_key":     entry["run_key"],
                    "trial_hash":  entry["trial_hash"],
                    "probes":      "+".join(entry["probes"]),
                    "results_csv": str(results_csv),
                    "task_type":   entry["task_type"],
                    "n_classes":   entry["n_classes"],
                })
    return pd.DataFrame(rows)

In [ ]:


# Discover every extraction artifact on disk, then keep only the pairs whose
# dataset matches a contract we have defined.
KNOWN_DATASETS = set(DATASETS.keys())   # amazon_polarity, emotion, goemo, isear, sst2, tweet_eval_emotion
REPEATS = 4
MAX_SAMPLES = None
VERBOSE = True
EXPERIMENT_ID = "main_run"


pairs = discover_extraction_pairs()
print(f"Extraction artifacts found : {len(pairs)}")
print(pairs.groupby("model").size().to_string())

entries = []
skipped_unknown = []
for row in pairs.itertuples(index=False):
    if row.dataset not in KNOWN_DATASETS:
        skipped_unknown.append((row.model, row.dataset))
        continue
    task_type = "multi_label" if row.dataset == "goemo" else "single_label"
    entries.append({
        "model":        row.model,
        "dataset":      row.dataset,
        "artifact_dir": row.artifact_dir,
        "contract":     goemotions_contract if task_type == "multi_label" else isear_contract,
        "dataset_df":   DATASETS[row.dataset],
    })

print(f"\nQueued for probing : {len(entries)}")
if skipped_unknown:
    print(f"Skipped {len(skipped_unknown)} pairs with no contract:")
    for m, d in skipped_unknown[:5]:
        print(f"  · {m} / {d}")

full_results = probe.run_matrix(
    entries,
    experiment_id=EXPERIMENT_ID,
    probes=probes,
    repeats=REPEATS,
    max_samples=MAX_SAMPLES,
    verbose=VERBOSE,
    checkpoint_dir=PROBE_ROOT / "_matrix_checkpoint",
    shuffled_label_control=True,
    shuffled_control_repeats=3,
)
print(f"\nMatrix completed. Full results shape: {full_results.shape}")

This custom, single_multi label should either go or imlemented, it's fine to use specific dataset and probe contracts for now, but the method needs to be regulated and one main schema to be produced. 

In [ ]:
# On any subsequent session, to load everything that's been probed so far:
runs = discover_probe_runs()
if runs.empty:
    print("No completed probe runs yet.")
else:
    # The one canonical column set is defined by Probe.py's output; concat directly.
    df = pd.concat([pd.read_csv(p) for p in runs["results_csv"]], ignore_index=True)
    print(f"Loaded {len(df):,} probe rows across {runs['run_key'].nunique()} runs.")

In [ ]:
if not full_results.empty:
    # Best layer per probe/model/dataset (highest test_macro_f1)
    best_per_probe = full_results.loc[full_results.groupby(["probe", "model", "dataset"])["test_macro_f1"].idxmax()]
    display_title("Best Layer per Probe (Macro-F1)")
    display(best_per_probe[["probe", "model", "dataset", "layer_index", "test_macro_f1", "probe_score"]])

    # Pivot table: model vs best macro-F1 per probe
    pivot_best = best_per_probe.pivot_table(index=["model", "dataset"], columns="probe", values="test_macro_f1")
    display_title("Best Macro-F1 Matrix")
    display(pivot_best.style.background_gradient(cmap='viridis', axis=None))

In [ ]:
output_plots_dir = Path("probe_plots")
output_plots_dir.mkdir(exist_ok=True)

# Use the plotting function from the probe module
probe.plot_full_dashboard(full_results, output_plots_dir)

In [ ]:
# Per model/dataset layer curves
for (model, dataset), group in full_results.groupby(["model", "dataset"]):
    plt.figure(figsize=(12, 6))
    for probe_name in group["probe"].unique():
        sub = group[group["probe"] == probe_name].sort_values("layer_index")
        plt.plot(sub["layer_index"], sub["test_macro_f1"], marker='o', label=probe_name)
    plt.title(f"{model} / {dataset} – Layer-wise Macro-F1")
    plt.xlabel("Layer index")
    plt.ylabel("Macro-F1")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()